In [ ]:
# @title 1. Install Dependencies
!pip install -q pandas numpy pyarrow pillow tqdm scikit-learn
!pip install -q transformers accelerate
# Installing faiss-gpu compatible with Colab's CUDA version
!pip install -q faiss-gpu-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 16.1 MB/s eta 0:00:00


In [ ]:
# @title 2. Configuration & Setup
import os

class Config:
    # Paths
    DRIVE_MOUNT_PATH = '/content/drive'
    BASE_DIR = os.path.join(DRIVE_MOUNT_PATH, 'MyDrive/RAGSearch')
    PRODUCT_DATA_PATH = os.path.join(BASE_DIR, 'preprocessed_products.parquet')
    QUERY_DATA_PATH = os.path.join(BASE_DIR, 'preprocessed_queries.parquet')

    # Image Paths: Source (Drive) -> Target (Local Runtime)
    DRIVE_IMAGE_DIR = os.path.join(BASE_DIR, 'balanced_images/')
    IMAGE_DIR = '/content/balanced_images/'  # Local NVMe storage

    # HF Token (Replace with your actual token)
    HF_TOKEN = "-"

    # Models to Evaluate
    MODELS = {
        "clip": "openai/clip-vit-base-patch16",
        "convnextv2": "facebook/convnextv2-base-22k-224",
        "siglip2": "google/siglip2-so400m-patch16-naflex",
        "dinov3": "facebook/dinov3-vits16-pretrain-lvd1689m"
    }

    # Data Splitting & sampling
    DATA_SAMPLE_FRACTION = 1.0  # Percentage of query data to use (0.0 to 1.0)
    TEST_SIZE = 0.5             # Validation/Test split
    SEED = 42

    # Augmentation
    AUGMENTATION_K = 5          # Number of "shown" products to inject per query group

    # Retrieval Config
    BATCH_SIZE = 64             # Batch size for encoding
    K_VALUES = [3, 5, 10, 20, 40]

    # Hardware
    DEVICE = "cuda"

config = Config()

# Mount Google Drive
from google.colab import drive
if not os.path.exists(config.DRIVE_MOUNT_PATH):
    drive.mount(config.DRIVE_MOUNT_PATH)

print(f"Configuration loaded. Images will be copied to: {config.IMAGE_DIR}")

Mounted at /content/drive
Configuration loaded. Images will be copied to: /content/balanced_images/


In [ ]:
# @title 3. Imports & Helper Functions
import sys
import gc
import time
import random
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from collections import Counter
import faiss

# Set seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(config.SEED)

def log(msg):
    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{timestamp}] {msg}")

def clean_memory():
    gc.collect()
    torch.cuda.empty_cache()

# Login to Hugging Face
from huggingface_hub import login
try:
    login(token=config.HF_TOKEN)
    log("Successfully logged in to Hugging Face.")
except Exception as e:
    log(f"Warning: HF Login failed or skipped. Ensure token is correct. Error: {e}")

[2026-01-04 11:40:12] Successfully logged in to Hugging Face.


In [ ]:
# @title 4 & 5. Data Processing, Augmentation & Splitting
import pandas as pd
import numpy as np
import glob
import os
import shutil
import random
from collections import Counter
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# --- 0. COPY IMAGES TO LOCAL RUNTIME (SPEEDUP) ---
log("Checking local image cache...")
if not os.path.exists(config.IMAGE_DIR):
    log(f"Copying images from Drive ({config.DRIVE_IMAGE_DIR}) to Local ({config.IMAGE_DIR})...")
    log("This may take 1-2 minutes but will speed up encoding by 20x.")
    try:
        shutil.copytree(config.DRIVE_IMAGE_DIR, config.IMAGE_DIR)
        log("Copy complete.")
    except Exception as e:
        log(f"Error copying images: {e}")
        # Fallback to drive if copy fails
        config.IMAGE_DIR = config.DRIVE_IMAGE_DIR
else:
    log("Images already exist locally. Skipping copy.")

# --- 1. LOAD DATA ---
log("Loading data...")
products_df = pd.read_parquet(config.PRODUCT_DATA_PATH)
queries_df = pd.read_parquet(config.QUERY_DATA_PATH)

# Type Enforcement
products_df['random_key'] = products_df['random_key'].astype(str).str.strip()

# --- 2. IMAGE INDEXING ---
log("Scanning local image directory...")
available_images = set()
image_files = glob.glob(os.path.join(config.IMAGE_DIR, "*.jpg"))
# Fast set building
for img_path in tqdm(image_files, desc="Indexing Images"):
    filename = os.path.basename(img_path)
    key = os.path.splitext(filename)[0].strip()
    available_images.add(key)

log(f"Images Found: {len(available_images)}")

# Filter Products to only those with images
products_df = products_df[products_df['random_key'].isin(available_images)].copy()
valid_product_keys = set(products_df['random_key'].values)
product_cat_map = products_df.set_index('random_key')['category_id'].to_dict()
log(f"Valid Products in DB: {len(products_df)}")

# --- 3. FILTER & CLEAN QUERY DATA ---
log("Cleaning query logs...")

def get_valid_keys(keys_array):
    if not isinstance(keys_array, (list, np.ndarray)):
        return []
    # Cast to string, strip, check existence
    valid = []
    for k in keys_array:
        k_str = str(k).strip()
        if k_str in valid_product_keys:
            valid.append(k_str)
    return list(set(valid))

# Filter Clicks (Viewed)
queries_df['valid_clicks'] = queries_df['viewed_base_products'].apply(get_valid_keys)
# Filter Shown (Result Base) - needed for augmentation
queries_df['valid_shown'] = queries_df['result_base_products'].apply(get_valid_keys)

# Drop rows with NO valid clicks (Primary task depends on clicks)
queries_df = queries_df[queries_df['valid_clicks'].apply(len) > 0].reset_index(drop=True)

# Normalize Query Text
queries_df['cleaned_query_norm'] = queries_df['cleaned_query'].astype(str).str.lower().str.strip()

log(f"Refined Query Sessions: {len(queries_df)}")

# --- 4. STRATIFICATION & SAMPLING ---
if config.DATA_SAMPLE_FRACTION < 1.0:
    log(f"Subsampling {config.DATA_SAMPLE_FRACTION*100}%...")
    # Infer Category
    def infer_category(row):
        if row['category_id'] != 0: return row['category_id']
        cats = [product_cat_map.get(k, 0) for k in row['valid_clicks'] if product_cat_map.get(k, 0) != 0]
        if not cats: return 0
        return Counter(cats).most_common(1)[0][0]

    queries_df['strat_category'] = queries_df.apply(infer_category, axis=1)

    try:
        queries_df, _ = train_test_split(queries_df, train_size=config.DATA_SAMPLE_FRACTION, stratify=queries_df['strat_category'], random_state=config.SEED)
    except:
        queries_df = queries_df.sample(frac=config.DATA_SAMPLE_FRACTION, random_state=config.SEED)

# --- 5. GROUPING & AUGMENTATION ---
log("Grouping queries and creating Pure vs Augmented datasets...")

# Aggregation: Group by text, union the product sets
grouped = queries_df.groupby('cleaned_query_norm').agg({
    'valid_clicks': lambda x: list(set([item for sublist in x for item in sublist])),
    'valid_shown': lambda x: list(set([item for sublist in x for item in sublist]))
})

pure_data = []
augmented_data = []

stats = {'pure_pairs': 0, 'aug_pairs': 0, 'pure_groups': 0, 'aug_groups': 0}

for text_query, row in tqdm(grouped.iterrows(), total=len(grouped), desc="Generating Pairs"):
    clicks = set(row['valid_clicks'])
    shown = set(row['valid_shown'])

    # 1. Pure Dataset Generation (Clicks only)
    if len(clicks) >= 2:
        stats['pure_groups'] += 1
        click_list = list(clicks)
        for i, query_key in enumerate(click_list):
            ground_truth = [p for j, p in enumerate(click_list) if i != j]
            pure_data.append({
                'query_key': query_key,
                'ground_truth': ground_truth,
                'original_text_query': text_query
            })
            stats['pure_pairs'] += 1

    # 2. Augmented Dataset Generation (Clicks + Random Shown)
    # Strategy: Start with clicks. Add up to K random items from (Shown - Clicks).
    # Then consider ALL of these as valid "relatives" for the task.
    candidates_to_add = list(shown - clicks)
    k_to_add = min(len(candidates_to_add), config.AUGMENTATION_K)

    augmented_set = list(clicks)
    if k_to_add > 0:
        random.seed(config.SEED + len(text_query)) # Deterministic randomness
        augmented_set += random.sample(candidates_to_add, k_to_add)

    if len(augmented_set) >= 2:
        stats['aug_groups'] += 1
        for i, query_key in enumerate(augmented_set):
            ground_truth = [p for j, p in enumerate(augmented_set) if i != j]
            augmented_data.append({
                'query_key': query_key,
                'ground_truth': ground_truth,
                'original_text_query': text_query
            })
            stats['aug_pairs'] += 1

log("Dataset Generation Stats:")
log(f"Pure (Original): {stats['pure_groups']} groups -> {stats['pure_pairs']} pairs")
log(f"Augmented:     {stats['aug_groups']} groups -> {stats['aug_pairs']} pairs")

# --- 6. SPLITTING ---
# Split based on unique text queries to prevent leakage
unique_texts = list(grouped.index)
train_texts, test_texts = train_test_split(unique_texts, test_size=config.TEST_SIZE, random_state=config.SEED)

pure_df = pd.DataFrame(pure_data)
aug_df = pd.DataFrame(augmented_data)

# Create Splits
# We check if dfs are not empty to avoid errors
if not pure_df.empty:
    val_pure_df = pure_df[pure_df['original_text_query'].isin(train_texts)].copy()
    test_pure_df = pure_df[pure_df['original_text_query'].isin(test_texts)].copy()
else:
    val_pure_df, test_pure_df = pd.DataFrame(), pd.DataFrame()

if not aug_df.empty:
    val_aug_df = aug_df[aug_df['original_text_query'].isin(train_texts)].copy()
    test_aug_df = aug_df[aug_df['original_text_query'].isin(test_texts)].copy()
else:
    val_aug_df, test_aug_df = pd.DataFrame(), pd.DataFrame()

log(f"Validation Pure: {len(val_pure_df)} | Validation Augmented: {len(val_aug_df)}")
log(f"Test Pure:       {len(test_pure_df)} | Test Augmented:       {len(test_aug_df)}")

# Prepare Candidate Pool (Shared)
candidate_products_df = products_df.reset_index(drop=True)
candidate_keys_list = list(candidate_products_df['random_key'].values)
candidate_keys_map = {i: k for i, k in enumerate(candidate_keys_list)}
log(f"Candidate Pool Size: {len(candidate_products_df)}")

[2026-01-04 11:40:30] Checking local image cache...
[2026-01-04 11:40:30] Copying images from Drive (/content/drive/MyDrive/RAGSearch/balanced_images/) to Local (/content/balanced_images/)...
[2026-01-04 11:40:30] This may take 1-2 minutes but will speed up encoding by 20x.
[2026-01-04 11:44:15] Copy complete.
[2026-01-04 11:44:15] Loading data...
[2026-01-04 11:44:28] Scanning local image directory...


Indexing Images:   0%|          | 0/9151 [00:00<?, ?it/s]

[2026-01-04 11:44:28] Images Found: 9151
[2026-01-04 11:44:28] Valid Products in DB: 9151
[2026-01-04 11:44:28] Cleaning query logs...
[2026-01-04 11:44:30] Refined Query Sessions: 1711
[2026-01-04 11:44:30] Grouping queries and creating Pure vs Augmented datasets...


Generating Pairs:   0%|          | 0/1249 [00:00<?, ?it/s]

[2026-01-04 11:44:30] Dataset Generation Stats:
[2026-01-04 11:44:30] Pure (Original): 67 groups -> 146 pairs
[2026-01-04 11:44:30] Augmented:     550 groups -> 1577 pairs
[2026-01-04 11:44:30] Validation Pure: 78 | Validation Augmented: 825
[2026-01-04 11:44:30] Test Pure:       68 | Test Augmented:       752
[2026-01-04 11:44:30] Candidate Pool Size: 9151


In [ ]:
# @title 6. Model Wrapper Class
from transformers import AutoImageProcessor, AutoModel, CLIPModel, CLIPProcessor, AutoProcessor
import torch
import torch.nn.functional as F

class EncoderFactory:
    def __init__(self, model_key, model_name):
        self.model_key = model_key
        self.model_name = model_name
        self.device = config.DEVICE
        self.model = None
        self.processor = None

    def load(self):
        log(f"Loading {self.model_name}...")

        try:
            if "siglip" in self.model_key:
                # SigLIP2: Load FULL model to access get_image_features API
                self.model = AutoModel.from_pretrained(self.model_name, token=config.HF_TOKEN, trust_remote_code=True)
                self.processor = AutoProcessor.from_pretrained(self.model_name, token=config.HF_TOKEN, trust_remote_code=True)
            elif "clip" in self.model_key:
                # CLIP: Loading vision_model directly works fine
                self.model = CLIPModel.from_pretrained(self.model_name, token=config.HF_TOKEN).vision_model
                self.processor = CLIPProcessor.from_pretrained(self.model_name, token=config.HF_TOKEN)
            elif "dinov3" in self.model_key:
                # DINOv3: Generic AutoModel
                self.model = AutoModel.from_pretrained(self.model_name, token=config.HF_TOKEN, trust_remote_code=True)
                self.processor = AutoImageProcessor.from_pretrained(self.model_name, token=config.HF_TOKEN, trust_remote_code=True)
            else:
                # ConvNext and others
                self.model = AutoModel.from_pretrained(self.model_name, token=config.HF_TOKEN, trust_remote_code=True)
                self.processor = AutoImageProcessor.from_pretrained(self.model_name, token=config.HF_TOKEN, trust_remote_code=True)

            self.model.to(self.device)
            self.model.eval()
            log("Model loaded successfully.")
        except Exception as e:
            log(f"Failed to load model {self.model_name}: {e}")
            raise e

    def encode(self, images):
        # images: List of PIL Images
        with torch.no_grad():
            inputs = self.processor(images=images, return_tensors="pt").to(self.device)

            if "siglip" in self.model_key:
                # SigLIP specific API (handles pixel_attention_mask)
                embeddings = self.model.get_image_features(**inputs)
            else:
                # Generic Forward Pass
                outputs = self.model(**inputs)

                # Extract embeddings based on architecture
                if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
                    embeddings = outputs.pooler_output
                elif hasattr(outputs, 'last_hidden_state'):
                    # Mean pooling or CLS token
                    embeddings = outputs.last_hidden_state[:, 0, :]
                else:
                    embeddings = outputs[0]

            # Normalize (Critical for Cosine Similarity)
            embeddings = F.normalize(embeddings, p=2, dim=1)

        return embeddings.cpu().numpy()

    def cleanup(self):
        del self.model
        del self.processor
        clean_memory()
        log(f"Unloaded {self.model_name}")

def load_image_batch(keys):
    batch_imgs = []
    valid_indices = []
    for i, k in enumerate(keys):
        path = os.path.join(config.IMAGE_DIR, f"{k}.jpg")
        try:
            img = Image.open(path).convert('RGB')
            batch_imgs.append(img)
            valid_indices.append(i)
        except Exception:
            pass
    return batch_imgs, valid_indices

In [ ]:
# @title 7. Evaluation Metrics
def calculate_metrics(retrieved_ids, ground_truth_ids, k_values):
    # retrieved_ids: List of list of strings (keys)
    # ground_truth_ids: List of list of strings (keys)

    metrics = {k: {'precision': [], 'recall': [], 'f1': []} for k in k_values}
    metrics['mrr'] = []
    metrics['ndcg'] = []

    for retrieved, gt in zip(retrieved_ids, ground_truth_ids):
        gt_set = set(gt)
        if not gt_set:
            continue

        # MRR
        first_correct_rank = 0
        for i, r in enumerate(retrieved):
            if r in gt_set:
                first_correct_rank = i + 1
                break
        metrics['mrr'].append(1.0 / first_correct_rank if first_correct_rank > 0 else 0.0)

        # NDCG (Binary relevance)
        dcg = 0.0
        idcg = 0.0
        for i, r in enumerate(retrieved):
            if r in gt_set:
                dcg += 1.0 / np.log2(i + 2)

        for i in range(min(len(retrieved), len(gt_set))):
            idcg += 1.0 / np.log2(i + 2)

        metrics['ndcg'].append(dcg / idcg if idcg > 0 else 0.0)

        # P, R, F1 @ K
        for k in k_values:
            top_k = set(retrieved[:k])
            hits = len(top_k.intersection(gt_set))

            p = hits / k
            r = hits / len(gt_set)
            f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0

            metrics[k]['precision'].append(p)
            metrics[k]['recall'].append(r)
            metrics[k]['f1'].append(f1)

    # Average out
    final_results = {}
    final_results['MRR'] = np.mean(metrics['mrr'])
    final_results['NDCG'] = np.mean(metrics['ndcg'])

    for k in k_values:
        final_results[f'P@{k}'] = np.mean(metrics[k]['precision'])
        final_results[f'R@{k}'] = np.mean(metrics[k]['recall'])
        final_results[f'F1@{k}'] = np.mean(metrics[k]['f1'])

    return final_results

In [ ]:
# @title 8. Main Pipeline Execution
if 'val_aug_df' not in locals():
    raise ValueError("CRITICAL: Dataframes not found. Ensure Data Pipeline ran successfully.")

results_log = []
checkpoint_path = os.path.join(config.BASE_DIR, "results_checkpoint.csv")

for model_key, model_name in config.MODELS.items():
    log(f"\n==================================================")
    log(f"Processing Model: {model_key} ({model_name})")

    # 1. Initialize Model
    try:
        encoder = EncoderFactory(model_key, model_name)
        encoder.load()
    except Exception as e:
        log(f"Skipping {model_key} due to load error: {e}")
        continue

    # 2. Build Index
    d = None
    index = None
    res = faiss.StandardGpuResources()

    log("Building Faiss Index...")
    chunk_size = config.BATCH_SIZE
    total_candidates = len(candidate_keys_list)

    # Batch Encode Candidates
    for i in tqdm(range(0, total_candidates, chunk_size), desc="Indexing"):
        chunk_keys = candidate_keys_list[i : i + chunk_size]
        imgs, valid_idxs = load_image_batch(chunk_keys)

        if not imgs: continue

        embs = encoder.encode(imgs)

        if index is None:
            d = embs.shape[1]
            cpu_index = faiss.IndexFlatIP(d)
            index = faiss.index_cpu_to_gpu(res, 0, cpu_index)

        index.add(embs)
        del imgs, embs

    log(f"Index built with {index.ntotal} vectors.")

    # 3. Evaluation Function
    def run_evaluation(eval_df, stage_name):
        if eval_df.empty:
            log(f"Skipping {stage_name} (Empty Data)")
            return None

        log(f"Evaluating on {stage_name} ({len(eval_df)} queries)...")
        query_keys = eval_df['query_key'].values
        ground_truths = eval_df['ground_truth'].values
        all_retrieved_keys = []

        for i in tqdm(range(0, len(query_keys), chunk_size), desc=f"Querying {stage_name}"):
            q_keys_batch = query_keys[i : i + chunk_size]
            q_imgs, valid_idxs = load_image_batch(q_keys_batch)

            if not q_imgs:
                for _ in range(len(q_keys_batch)): all_retrieved_keys.append([])
                continue

            q_embs = encoder.encode(q_imgs)

            k_max = max(config.K_VALUES)
            D, I = index.search(q_embs, k_max + 1)

            for j, result_indices in enumerate(I):
                query_k = q_keys_batch[valid_idxs[j]]
                retrieved = [candidate_keys_map[idx] for idx in result_indices if idx != -1]
                retrieved = [r for r in retrieved if r != query_k][:k_max]
                all_retrieved_keys.append(retrieved)

        metrics = calculate_metrics(all_retrieved_keys, ground_truths, config.K_VALUES)
        metrics['model'] = model_key
        metrics['stage'] = stage_name

        log(f"{stage_name} -> NDCG: {metrics['NDCG']:.4f} | MRR: {metrics['MRR']:.4f}")
        return metrics

    # Run Eval on PURE and AUGMENTED
    metrics_pure = run_evaluation(val_pure_df, "Val_Pure")
    if metrics_pure:
        results_log.append(metrics_pure)
        # SAVE IMMEDIATELY
        pd.DataFrame(results_log).to_csv(checkpoint_path, index=False)

    metrics_aug = run_evaluation(val_aug_df, "Val_Augmented")
    if metrics_aug:
        results_log.append(metrics_aug)
        # SAVE IMMEDIATELY
        pd.DataFrame(results_log).to_csv(checkpoint_path, index=False)

    # Cleanup
    del index
    encoder.cleanup()
    clean_memory()

# 4. Model Selection
log("\n==================================================")
log("Model Selection")
results_df = pd.DataFrame(results_log)

if 'Val_Augmented' in results_df['stage'].values:
    selection_stage = 'Val_Augmented'
else:
    selection_stage = 'Val_Pure'

log(f"Selecting best model based on stage: {selection_stage}")
if not results_df.empty:
    best_model_row = results_df[results_df['stage'] == selection_stage].sort_values(by='NDCG', ascending=False).iloc[0]
    best_model_key = best_model_row['model']
    log(f"Best Model: {best_model_key} (NDCG: {best_model_row['NDCG']:.4f})")

    print("\n--- VALIDATION RESULTS ---")
    print(results_df[results_df['stage'].isin(['Val_Pure', 'Val_Augmented'])])
else:
    log("No results generated. Using first model as fallback.")
    best_model_key = list(config.MODELS.keys())[0]

# 5. Final Test
log(f"\nRunning Final Test on {best_model_key}...")
encoder = EncoderFactory(best_model_key, config.MODELS[best_model_key])
encoder.load()

# Rebuild Index
res = faiss.StandardGpuResources()
index = None
chunk_size = config.BATCH_SIZE
for i in tqdm(range(0, len(candidate_keys_list), chunk_size), desc="Re-indexing"):
    chunk_keys = candidate_keys_list[i : i + chunk_size]
    imgs, _ = load_image_batch(chunk_keys)
    if not imgs: continue
    embs = encoder.encode(imgs)
    if index is None:
        cpu_index = faiss.IndexFlatIP(embs.shape[1])
        index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
    index.add(embs)
    del imgs, embs

# Eval Test Pure
test_pure_metrics = run_evaluation(test_pure_df, "Test_Pure")
if test_pure_metrics:
    test_pure_metrics['model'] = best_model_key
    print("\n--- FINAL TEST RESULTS (PURE) ---")
    print(pd.Series(test_pure_metrics))

# Eval Test Augmented
test_aug_metrics = run_evaluation(test_aug_df, "Test_Augmented")
if test_aug_metrics:
    test_aug_metrics['model'] = best_model_key
    print("\n--- FINAL TEST RESULTS (AUGMENTED) ---")
    print(pd.Series(test_aug_metrics))

# Final Save
pd.DataFrame([test_pure_metrics, test_aug_metrics]).to_csv(os.path.join(config.BASE_DIR, "final_test_results.csv"), index=False)

[2026-01-04 11:45:36] 
[2026-01-04 11:45:36] Processing Model: clip (openai/clip-vit-base-patch16)
[2026-01-04 11:45:36] Loading openai/clip-vit-base-patch16...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/599M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

...

[2026-01-04 12:25:31] Evaluating on Test_Pure (68 queries)...


Querying Test_Pure:   0%|          | 0/2 [00:00<?, ?it/s]

[2026-01-04 12:25:36] Test_Pure -> NDCG: 0.2884 | MRR: 0.2095

--- FINAL TEST RESULTS (PURE) ---
MRR       0.209492
NDCG      0.288407
P@3       0.093137
R@3       0.257353
F1@3      0.135294
P@5       0.076471
R@5       0.338235
F1@5      0.123249
P@10      0.042647
R@10      0.382353
F1@10     0.076203
P@20      0.026471
R@20      0.477941
F1@20     0.049975
P@40      0.016912
R@40      0.595588
F1@40     0.032811
model      siglip2
stage    Test_Pure
dtype: object
[2026-01-04 12:25:36] Evaluating on Test_Augmented (752 queries)...


Querying Test_Augmented:   0%|          | 0/12 [00:00<?, ?it/s]

[2026-01-04 12:26:28] Test_Augmented -> NDCG: 0.2762 | MRR: 0.2609

--- FINAL TEST RESULTS (AUGMENTED) ---
MRR            0.260883
NDCG           0.276242
P@3            0.113032
R@3            0.174357
F1@3           0.125896
P@5            0.090691
R@5            0.231206
F1@5           0.121231
P@10           0.063697
R@10            0.31844
F1@10          0.100789
P@20           0.043218
R@20           0.410527
F1@20          0.075615
P@40           0.027593
R@40           0.507957
F1@40          0.051373
model           siglip2
stage    Test_Augmented
dtype: object
